# Quantization


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'course_tools').exists() or (path / 'picollm').exists()), Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

LECTURE_NOTE_TITLE = 'Quantization'
print(f'Lecture note: {LECTURE_NOTE_TITLE}')


Lecture note: Quantization


## Demo A: Memory by dtype


In [2]:
def bytes_used(num_params: int, bits_per_value: int) -> float:
    return num_params * bits_per_value / 8 / (1024 ** 3)

num_params = 1_500_000_000
for name, bits in [('fp32', 32), ('fp16', 16), ('int8', 8), ('int4', 4)]:
    print(name, round(bytes_used(num_params, bits), 2), 'GB')


fp32 5.59 GB
fp16 2.79 GB
int8 1.4 GB
int4 0.7 GB


## Demo B: Toy int8 quantization


In [3]:
def symmetric_int8_quantize(x: torch.Tensor):
    scale = float(x.abs().max() / 127.0) if float(x.abs().max()) > 0 else 1.0
    q = torch.clamp(torch.round(x / scale), -127, 127).to(torch.int8)
    return q, scale

x = torch.tensor([[-1.25, -0.25, 0.0, 0.8], [1.5, 0.3, -0.7, 0.12]], dtype=torch.float32)
q, scale = symmetric_int8_quantize(x)
restored = q.float() * scale
print('original:\n', x)
print('\nquantized:\n', q)
print('\nrestored:\n', restored)
print('\nmean abs error:', float((x - restored).abs().mean()))


original:
 tensor([[-1.2500, -0.2500,  0.0000,  0.8000],
        [ 1.5000,  0.3000, -0.7000,  0.1200]])

quantized:
 tensor([[-106,  -21,    0,   68],
        [ 127,   25,  -59,   10]], dtype=torch.int8)

restored:
 tensor([[-1.2520, -0.2480,  0.0000,  0.8031],
        [ 1.5000,  0.2953, -0.6969,  0.1181]])

mean abs error: 0.002106289379298687


## Demo C: Platform-aware serving choices


In [4]:
import warnings

warnings.filterwarnings('ignore', message='IProgress not found.*')
from picollm.common.device import summarize_device

for preferred in ['auto', 'cpu', 'mps', 'cuda']:
    print(preferred, summarize_device(preferred))


auto {'requested': 'auto', 'resolved': 'mps', 'dtype': 'float16', 'platform': 'macOS-26.3.1-arm64-arm-64bit', 'cuda_available': 'false', 'mps_available': 'true'}
cpu {'requested': 'cpu', 'resolved': 'cpu', 'dtype': 'float32', 'platform': 'macOS-26.3.1-arm64-arm-64bit', 'cuda_available': 'false', 'mps_available': 'true'}
mps {'requested': 'mps', 'resolved': 'mps', 'dtype': 'float16', 'platform': 'macOS-26.3.1-arm64-arm-64bit', 'cuda_available': 'false', 'mps_available': 'true'}
cuda {'requested': 'cuda', 'resolved': 'cuda', 'dtype': 'float16', 'platform': 'macOS-26.3.1-arm64-arm-64bit', 'cuda_available': 'false', 'mps_available': 'true'}


## Demo D: Where quantization appears in the serious model path


In [5]:
serve_flags = {
    'Mac': '--device mps --quantization none',
    'Linux CUDA': '--device cuda --quantization 4bit',
    'Windows CUDA': '--device cuda --quantization 8bit',
    'CPU only': '--device cpu --quantization none',
}
serve_flags


{'Mac': '--device mps --quantization none',
 'Linux CUDA': '--device cuda --quantization 4bit',
 'Windows CUDA': '--device cuda --quantization 8bit',
 'CPU only': '--device cpu --quantization none'}

## Compare with nanochat


In [6]:
notes = {
    'course repo': 'teach the storage/accuracy tradeoff first',
    'nanochat': 'goes much deeper into optimized kernels and hardware-efficient training/inference choices',
}
notes


{'course repo': 'teach the storage/accuracy tradeoff first',
 'nanochat': 'goes much deeper into optimized kernels and hardware-efficient training/inference choices'}